# rs-aos-stats: Hit Rules Comparison

This notebook compares different "Hit Rules" (Normal, Critical Auto-Wound, Critical Mortal Wound, Critical Double Hit) and plots their damage distributions.

In [1]:
import rs_aos_stats
from rs_aos_stats import (
    AttackStats, DefenseStats, CombatConfig, compute_damages,
)
import matplotlib.pyplot as plt
import numpy as np
import ipywidgets as widgets
from ipywidgets import interact, interactive_output, HBox, VBox, Label, Layout

## Interactive Plot

Use the sliders below to adjust the attack and defense characteristics.

In [ ]:
def plot_comparison(attacks, to_hit, to_wound, rend, damage, save, ward):
    # Setup Stats
    attack_stats = AttackStats(attacks, to_hit, to_wound, rend, damage)
    
    # Ward of 7 means "None" for this example
    ward_val = ward if ward < 7 else None
    defense_stats = DefenseStats(save, ward_val)
    
    config = CombatConfig(attack_stats, defense_stats, None)
    
    # Define Scenarios
    scenarios = {
        "Normal": "normal",
        "Crit Auto-Wound": "crit_auto_wound",
        "Crit MW": "crit_mortal_wound",
        "Crit Double Hit": "crit_double_hit",
    }
    
    results = {}
    
    # Compute for each scenario
    for name, hit_rule_type in scenarios.items():
        damages_dist = compute_damages(config, hit_rule_type)
        results[name] = damages_dist
    
    # Plot
    plt.figure(figsize=(12, 6))
    
    # Find global max damage to set x-axis range
    all_damages = []
    for data in results.values():
        all_damages.extend([d[0] for d in data])
    max_damage = max(all_damages) if all_damages else 0
    
    # Define bar width and positions
    n_scenarios = len(results)
    # Reduced width to increase spacing between groups
    bar_width = 0.6 / n_scenarios
    
    for i, (name, data) in enumerate(results.items()):
        # Create a dictionary for easy lookup of probability by damage
        prob_map = {d[0]: d[1] for d in data}
        
        # Prepare x and y for plotting, ensuring we cover the range
        x = np.arange(max_damage + 1)
        y = [prob_map.get(d, 0) for d in x]
        
        # Calculate offset positions
        positions = x + (i - n_scenarios/2 + 0.5) * bar_width
        
        mean_damage = sum(d * p for d, p in data)
        plt.bar(positions, y, width=bar_width, label=f"{name} (Mean: {mean_damage:.2f})")
            
    plt.title("Damage Probability Distribution")
    plt.xlabel("Damage Dealt")
    plt.ylabel("Probability")
    plt.legend()
    plt.grid(True, alpha=0.3, axis='y')
    
    # Force integer ticks on x-axis
    plt.xticks(np.arange(0, max_damage + 1, 1))
    plt.xlim(-0.5, max_damage + 1)
    
    plt.show()

# Create Widgets
style = {'description_width': 'initial'}

# Attack Widgets
w_attacks = widgets.IntSlider(min=1, max=30, step=1, value=10, description='Attacks:', style=style)
w_to_hit = widgets.IntSlider(min=2, max=6, step=1, value=3, description='To Hit (X+):', style=style)
w_to_wound = widgets.IntSlider(min=2, max=6, step=1, value=3, description='To Wound (X+):', style=style)
w_rend = widgets.IntSlider(min=0, max=5, step=1, value=1, description='Rend:', style=style)
w_damage = widgets.IntSlider(min=1, max=6, step=1, value=1, description='Damage:', style=style)

# Defense Widgets
w_save = widgets.IntSlider(min=2, max=6, step=1, value=4, description='Save (X+):', style=style)
w_ward = widgets.IntSlider(min=2, max=7, step=1, value=7, description='Ward (X+, 7=None):', style=style)

# Layout
attack_box = VBox([
    Label(value="<b>Attack Stats</b>"),
    w_attacks, w_to_hit, w_to_wound, w_rend, w_damage
], layout=Layout(border='1px solid #ccc', padding='10px', margin='0 10px 0 0'))

defense_box = VBox([
    Label(value="<b>Defense Stats</b>"),
    w_save, w_ward
], layout=Layout(border='1px solid #ccc', padding='10px'))

ui = HBox([attack_box, defense_box])

# Output
out = interactive_output(plot_comparison, {
    'attacks': w_attacks,
    'to_hit': w_to_hit,
    'to_wound': w_to_wound,
    'rend': w_rend,
    'damage': w_damage,
    'save': w_save,
    'ward': w_ward
})

display(ui, out)